## RAG Personal implementation

This notebook implements the main steps to build a Retrieval Augmented Generation System.

RAG's are commonly known because of its properties for answering questions about specific documents stored in a database called *base of knowledge*

The main steps in a RAG implementation are:
1. **Index:** Load the document into a vector store
2. **Search:** Build a custom tool that runs vector similarity search and finds the relevant pieces to answer the question
3. **Analyze:** Delegate the file analysis to a subagent that reads the file and returns a focused summary
4. **Synthesize:** Use the main agent to get the final answer from subagent reports

### The pipeline

In the indexing step, you’ll take the source content and convert chunks of it into numerical representations.

Indexing commonly works in four steps:
1. **Load:** Load your data sources into Document objects.
2. **Split:** Use text splitters to break large Documents into smaller chunks. This is useful both for indexing data and passing it to a model, as large chunks are harder to search over and either do not fit in a model’s finite context window or use more tokens than necessary.
3. **Embed:** Embeddings models convert each chunk into a numeric vector that captures its meaning, enabling similarity search over your content.
4. **Store:** Use a VectorStore to index chunks and their embeddings for retrieval.

#### Load

In [2]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "../docs/DECRETO 3022 DEL 27 DE DICIEMBRE DE 2013 - Aplicación NIIF.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

230


A page is often too coarse for retrieval. Split pages further so relevant passages are not diluted by surrounding text. RecursiveCharacterTextSplitter recursively splits on common separators (such as newlines) until each chunk is the target size. This is the recommended text splitter for generic text use cases.

Set add_start_index=True so each split keeps a start_index metadata field for its character offset in the original document.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

929


### Select the embeddings

In [ ]:
from dotenv import find_dotenv, load_dotenv

_ = load_dotenv(find_dotenv())

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

Select a Vector Store

In [6]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="../chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

NameError: name 'embeddings' is not defined